In [1]:
from google import genai
from pydantic import BaseModel
import enum
import json
from time import sleep

from utils import *

In [ ]:
PROMPT_TEMPLATE = '''Organize the following questions and answers into a structured json output. I will provide the values of the source name, year, and difficulty for you to fill in directly. You must fill out the remaining fields. 
For the question_id, you should construct a string in the format name_year_number, where number is the question number. If the question has multiple parts, break each part into its own question, and label its number alphabetically. For example, question 1 with three parts would become 1a, 1b, and 1c. If the later questions reference earlier questions, disambiguate any names or alter the question so that it stands alone.
Do not assign a difficulty level yourself. If the given difficulty is unknown, assign "unknown".
If a question is asked in English, you should assign the question_language as English. This includes who, what, when, where, why, and how questions. This also includes instructions, such as translate, give a synonym, etc. Only assign Latin if the question itself is asked in Latin, not only if Latin words are used in the question.
If the answer is a non-English name, assign the answer_language as Latin. Try to assign the language used in the majority of the answer, or in the most important part of the answer. Only label the language as "both" if the question specifically asked for an answer in both languages.
To determine the correctness logic, first determine if there are multiple possible answers in the "answers" field. If there's only one possible short answer, or this is a multiple choice question with only one answer, set the correctness logic to "na", since this is a single choice question. If there are multiple possible answers, consider what the question is asking. If the question is asking for a single answer, set the correctness logic to "any". If the question requires all of the gold answers to be correct, set the correctness logic to "all". If the question requires a certain number between 1 and the total number of gold answers to be correct (not inclusive), set the correctness logic to "n_of", and set the n_required field to the number of gold answers that must be correct.

Source name: {}
Source year: {}
Difficulty: {}

Questions:
{}
'''

In [2]:
source = 'NJCL-Certamen'
year = ''
difficulty = ''
questions = ''

In [3]:
with open('../ocr/gemini/gemini-key.txt', 'r') as f:
    api_key = f.read().strip()
client = genai.Client(api_key=api_key)

In [4]:
with open('../data/semi_structured/certamen_1996_2009.json', 'r') as f:
    data = json.load(f)

In [31]:
key = '1996_UNKNOWN'
questions = data[key][6]


year, difficulty = key.split('_')
prompt = PROMPT_TEMPLATE.format(source, year, difficulty, questions)
prompt

'Organize the following questions and answers into a structured json output. I will provide the values of the source name, year, and difficulty for you to fill in directly. You must fill out the remaining fields. \nFor the question_id, you should construct a string in the format name_year_number, where number is the question number. If the question has multiple parts, break each part into its own question, and label its number alphabetically. For example, question 1 with three parts would become 1a, 1b, and 1c. If the later questions reference earlier questions, disambiguate any names or alter the question so that it stands alone.\nIf a question is asked in English, you should assign the question_language as English. This includes who, what, when, where, why, and how questions. This also includes instructions, such as translate, give a synonym, etc. Only assign Latin if the question itself is asked in Latin, not only if Latin words are used in the question.\nIf the answer is a non-Engl

In [32]:

response = client.models.generate_content(
    model='gemini-2.0-flash',
    contents=prompt,
    config=config
)

In [38]:
type(eval(response.text))

list

In [5]:
save_file = '../data/structured/certamen_1996_2009.json'
all_response_jsons = []


In [6]:
with open(save_file, 'r') as f:
    all_response_jsons = json.load(f)

len(all_response_jsons)

4439

In [7]:
i = 0
for key in data:
    print('key:', key)
    if i < 5: 
        i += 1
        continue
    
    year, difficulty = key.split('_')
    
    #questions = '\n'.join(data[key])
    
    for question in data[key]:
        prompt = PROMPT_TEMPLATE.format(source, year, difficulty, question)
        response = client.models.generate_content(
            model='gemini-2.0-flash',
            contents=prompt,
            config=config
        )
        all_response_jsons += eval(response.text)
        sleep(0.2)

    with open(save_file, 'w') as f:
        json.dump(all_response_jsons, f, indent=4)

    i += 1


key: 1996_UNKNOWN
key: 1996_LOWER
key: 1996_NOVICE
key: 1996_UPPER
key: 1999_NOVICE
key: 2000_LOWER
key: 2002_LOWER
key: 2002_NOVICE
key: 2002_UPPER


In [8]:
len(all_response_jsons)

13213

In [46]:
with open(save_file, 'w') as f:
    json.dump(all_response_jsons, f, indent=4)